In [6]:
import json
import pandas as pd
import requests
from opensky_api import OpenSkyApi

In [ ]:
api = OpenSkyApi()
states = api.get_states(bbox=(36.5, 42.5, 25.5, 45.0)) # bbox = (min latitude, max latitude, min longitude, max longitude), Now setted for Türkiye's Coordinations.

json_list = []
columns = ["callsign",
         "icao24",
         "latitude",
         "longitude",
         "on_ground",
         "origin_country",
         "velocity",
         "baro_alt", "vertical_rate", "category", "geo_altitude"
         ]

for s in states.states:
  aircraft_data = {
         "callsign":s.callsign,
         "icao24":s.icao24,
         "latitude":s.latitude,
         "longitude":s.longitude,
         "on_ground":s.on_ground,
         "origin_country":s.origin_country,
         "velocity":s.velocity,
         "baro_alt":s.baro_altitude,
         "vertical_rate":s.vertical_rate,
         "category":s.category,
         "geo_altitude":s.geo_altitude
  }
  json_list.append(aircraft_data)

#output_json = json.dumps(json_list, indent=4)

df = pd.DataFrame(json_list, columns=columns)

cdf = df.copy()

cdf = cdf[cdf["on_ground"]==False].dropna(subset=["longitude","latitude","baro_alt"])

cdf["callsign"] = cdf["callsign"].str.strip()

cdf["airline_code"] = cdf["callsign"].str.extract(r'^([A-z]{3})')

cdf["velocity_knots"] = cdf["velocity"]*1.94384
cdf["velocity_knots"] = cdf["velocity_knots"].round(5)

cdf["vertical_rate_fpm"] = cdf["vertical_rate"]*196.85

cdf["baro_alt_fpm"] = cdf["baro_alt"]*3.28084
cdf["baro_alt_fpm"] = cdf["baro_alt"].round(5)
cdf

cdf.to_csv("temp_flights.csv", index=False)